# PhoBapunPaper FULL ARCHIVE — Post-Batch Across-Session Analysis

Analogous to `EXTERNAL/PhoDibaPaper2024Book/PhoDibaPaper2024_FULL_ARCHIVE_2026-08-31.ipynb`, but for **Bapun** sessions.

Loads underscore-named CSVs transferred from Great Lakes `collected_outputs` via the Bapun-specific loader (no KDIBA `-(file_type)` rename required).

Focus:
- Train/test held-out position decoding error
- TwoNovel context-decoder accuracy
- OpenField vs TwoNovel split

In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

from copy import deepcopy
from typing import Dict, List, Optional, Tuple
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from neuropy.utils.result_context import IdentifyingContext
from pyphocorehelpers.Filesystem.path_helpers import find_first_extant_path
from pyphocorehelpers.gui.Jupyter.simple_widgets import fullwidth_path_widget

from pyphoplacecellanalysis.SpecificResults.AcrossSessionResults import (
    load_bapun_across_sessions_exported_files,
    AcrossSessionHelpers,
)

pd.set_option('display.max_columns', 100)
debug_print: bool = True

## Paths / batch stamp

In [ ]:
TODAY_DAY_DATE: str = '2026-11-27_Apogee'
BATCH_DATE_TO_USE: str = '2026-11-27_Bapun_GL'
cuttoff_date: datetime = datetime(2026, 11, 1)

known_collected_outputs_paths = [Path(v).resolve() for v in [
    '/Users/pho/data/collected_outputs',
    '/Volumes/SwapSSD/Data/collected_outputs',
    r'K:/scratch/collected_outputs',
    '/Users/pho/Dropbox (University of Michigan)/MED-DibaLabDropbox/Data/Pho/Outputs/output/collected_outputs',
    r'C:/Users/pho/repos/Spike3DWorkEnv/Spike3D/output/collected_outputs',
    '/home/halechr/cloud/turbo/Data/Output/collected_outputs',
    '/home/halechr/cloud/turbo/Pho/Output/collected_outputs',
    '/home/halechr/FastData/collected_outputs/',
]]
collected_outputs_directory = find_first_extant_path(known_collected_outputs_paths)
assert collected_outputs_directory.exists(), f'collected_outputs_directory: {collected_outputs_directory} does not exist!'
print(f'collected_outputs_directory: {collected_outputs_directory}')
display(fullwidth_path_widget(collected_outputs_directory))

across_sessions_output_folder: Path = collected_outputs_directory.joinpath('../across_sessions').resolve()
across_sessions_output_folder.mkdir(parents=True, exist_ok=True)
figures_folder: Path = collected_outputs_directory.joinpath('figures').resolve()
figures_folder.mkdir(parents=True, exist_ok=True)
publication_figures_export_folder: Path = across_sessions_output_folder.joinpath(f'{BATCH_DATE_TO_USE}_bapun_figures').resolve()
publication_figures_export_folder.mkdir(parents=True, exist_ok=True)

## Expected batch session contexts (mirrors ProcessBatchOutputs_Bapun_Batch.ipy):
expected_session_contexts: List[IdentifyingContext] = [
    IdentifyingContext(format_name='bapun', animal='RatJ', session_name='Day4Openfield'),
    IdentifyingContext(format_name='bapun', animal='RatK', session_name='Day4Openfield'),
    IdentifyingContext(format_name='bapun', animal='RatN', session_name='Day4OpenField'),
    IdentifyingContext(format_name='bapun', animal='RatS', session_name='Day1OpenField'),
    IdentifyingContext(format_name='bapun', animal='RatS', session_name='Day4OpenField'),
    IdentifyingContext(format_name='bapun', animal='RatU', session_name='Day5Openfield'),
    IdentifyingContext(format_name='bapun', animal='RatJ', session_name='Day3TwoNovel'),
    IdentifyingContext(format_name='bapun', animal='RatK', session_name='Day3TwoNovel'),
]

## Load Bapun collected_outputs CSVs

Uses `load_bapun_across_sessions_exported_files` which understands:
- `*_bapun_train_test_decoder_error.csv` / `_agg.csv`
- `*_bapun_context_decoder_laps.csv` / `_summary_agg.csv`

In [ ]:
dfs_dict, most_recent_parsed_files_df = load_bapun_across_sessions_exported_files(
    collected_outputs_directory=collected_outputs_directory,
    cuttoff_date=cuttoff_date,
    debug_print=debug_print,
)

bapun_train_test_decoder_error_df: pd.DataFrame = dfs_dict['bapun_train_test_decoder_error_df']
bapun_train_test_decoder_error_agg_df: pd.DataFrame = dfs_dict['bapun_train_test_decoder_error_agg_df']
bapun_context_decoder_laps_df: pd.DataFrame = dfs_dict['bapun_context_decoder_laps_df']
bapun_context_decoder_summary_agg_df: pd.DataFrame = dfs_dict['bapun_context_decoder_summary_agg_df']

display(most_recent_parsed_files_df)
print({k: len(v) for k, v in dfs_dict.items()})

## Coverage / quality inventory

In [ ]:
coverage_df: pd.DataFrame = most_recent_parsed_files_df.copy() if len(most_recent_parsed_files_df) > 0 else pd.DataFrame(columns=['session_stem', 'file_type', 'experiment_type', 'animal', 'session_name'])
if len(coverage_df) > 0:
    coverage_pivot = (
        coverage_df.assign(present=True)
        .pivot_table(index=['session_stem', 'animal', 'session_name', 'experiment_type'], columns='file_type', values='present', aggfunc='any', fill_value=False)
        .reset_index()
    )
    display(coverage_pivot)
    display(coverage_df.groupby('experiment_type')['session_stem'].nunique().rename('n_sessions'))
else:
    coverage_pivot = pd.DataFrame()
    print('No Bapun collected_outputs CSVs found yet for this cutoff. After GL run + Globus transfer, re-run the load cell.')

## Compare against expected batch contexts (best-effort on session_name substring):
found_session_names = set(coverage_df['session_name'].dropna().astype(str).tolist()) if len(coverage_df) > 0 else set()
found_stems = set(coverage_df['session_stem'].dropna().astype(str).tolist()) if len(coverage_df) > 0 else set()
missing_contexts = []
for ctx in expected_session_contexts:
    sn = str(ctx.session_name)
    animal = str(ctx.animal)
    matched = any((sn in stem) or (stem.endswith(sn)) or (f'{animal}{sn}' == stem) or (f'{animal}-{sn}' == stem) for stem in found_stems)
    matched = matched or (sn in found_session_names)
    if not matched:
        missing_contexts.append(ctx)
## END for ctx in expected_session_contexts...
print(f'missing_expected_contexts ({len(missing_contexts)}): {missing_contexts}')

## Figure A — Train/test position error

Session × maze heatmap and box/strip of median abs error. Split OpenField vs TwoNovel.

In [ ]:
agg_df = deepcopy(bapun_train_test_decoder_error_agg_df)
err_df = deepcopy(bapun_train_test_decoder_error_df)

y_agg_col: str = 'abs_err_cm_median_median'
if len(agg_df) == 0 or (y_agg_col not in agg_df.columns):
    print(f'Skipping Figure A: agg_df empty or missing `{y_agg_col}` (cols={list(agg_df.columns)})')
else:
    for experiment_type in ['OpenField', 'TwoNovel']:
        sub_agg = agg_df[agg_df['experiment_type'] == experiment_type].copy()
        if len(sub_agg) == 0:
            print(f'No agg rows for experiment_type={experiment_type}')
            continue

        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        heat = sub_agg.pivot_table(index='session_stem', columns='maze', values=y_agg_col, aggfunc='mean')
        sns.heatmap(heat, annot=True, fmt='.1f', cmap='viridis', ax=axes[0])
        axes[0].set_title(f'{experiment_type}: session × maze {y_agg_col}')

        sns.boxplot(data=sub_agg, x='maze', y=y_agg_col, ax=axes[1])
        sns.stripplot(data=sub_agg, x='maze', y=y_agg_col, color='k', alpha=0.6, ax=axes[1])
        axes[1].set_title(f'{experiment_type}: {y_agg_col} by maze')
        fig.tight_layout()
        out_path = publication_figures_export_folder.joinpath(f'{BATCH_DATE_TO_USE}_{experiment_type}_train_test_abs_err.png')
        fig.savefig(out_path, dpi=200)
        print(f'saved: {out_path}')
        display(fig)
    ## END for experiment_type in ['OpenField', 'TwoNovel']...

    ## Optional 1D/2D comparison when columns exist:
    optional_metric_cols = [c for c in ['abs_err_cm_1D_median_median', 'abs_err_cm_2D_median_median'] if c in agg_df.columns]
    if len(optional_metric_cols) > 0:
        long_metrics = agg_df.melt(id_vars=['session_stem', 'maze', 'experiment_type'], value_vars=[y_agg_col] + optional_metric_cols, var_name='metric', value_name='value')
        g = sns.catplot(data=long_metrics, x='maze', y='value', hue='metric', col='experiment_type', kind='box', height=4, aspect=1.1)
        g.fig.suptitle('Train/test abs error metrics', y=1.02)
        out_path = publication_figures_export_folder.joinpath(f'{BATCH_DATE_TO_USE}_train_test_abs_err_metrics.png')
        g.fig.savefig(out_path, dpi=200, bbox_inches='tight')
        print(f'saved: {out_path}')

## Figure B — Context decoder (TwoNovel)

Overall + per-maze `% correct` from `bapun_context_decoder_summary_agg`.

In [ ]:
ctx_summary_df = deepcopy(bapun_context_decoder_summary_agg_df)
ctx_laps_df = deepcopy(bapun_context_decoder_laps_df)

if len(ctx_summary_df) == 0:
    print('Skipping Figure B: no context_decoder_summary_agg CSVs loaded (expected for OpenField-only or incomplete TwoNovel exports).')
else:
    display(ctx_summary_df)
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(data=ctx_summary_df, x='session_stem', y='percent_context_correct', hue='maze', ax=ax)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('percent_context_correct')
    ax.set_title('Bapun context decoder accuracy')
    plt.xticks(rotation=30, ha='right')
    fig.tight_layout()
    out_path = publication_figures_export_folder.joinpath(f'{BATCH_DATE_TO_USE}_context_decoder_accuracy.png')
    fig.savefig(out_path, dpi=200)
    print(f'saved: {out_path}')
    display(fig)

    if len(ctx_laps_df) > 0 and {'true_context', 'decoded_most_likely_context'}.issubset(ctx_laps_df.columns):
        confusion = pd.crosstab(ctx_laps_df['true_context'], ctx_laps_df['decoded_most_likely_context'], normalize='index')
        display(confusion)
        fig2, ax2 = plt.subplots(figsize=(5, 4))
        sns.heatmap(confusion, annot=True, fmt='.2f', cmap='Blues', ax=ax2, vmin=0, vmax=1)
        ax2.set_title('Context decoder confusion (row-normalized)')
        fig2.tight_layout()
        out_path2 = publication_figures_export_folder.joinpath(f'{BATCH_DATE_TO_USE}_context_decoder_confusion.png')
        fig2.savefig(out_path2, dpi=200)
        print(f'saved: {out_path2}')
        display(fig2)

## Figure C — Sorted vs clusterless inventory

Clusterless train/test CSVs are not always exported with the same underscore batch-date prefix. This cell inventories related figure/CSV artifacts when present.

In [ ]:
clusterless_csv_candidates = list(collected_outputs_directory.glob('*clusterless*bapun*train_test*.csv')) + list(collected_outputs_directory.glob('*bapun*clusterless*train_test*.csv'))
clusterless_fig_candidates = []
if figures_folder.exists():
    clusterless_fig_candidates = list(figures_folder.rglob('*bapun*clusterless*')) + list(figures_folder.rglob('*clusterless*train_test*'))
## END if figures_folder.exists()...

print(f'clusterless CSV candidates: {len(clusterless_csv_candidates)}')
for p in clusterless_csv_candidates[:20]:
    print(f'  {p.name}')
## END for p in clusterless_csv_candidates[:20]...

print(f'clusterless figure candidates: {len(clusterless_fig_candidates)}')
for p in clusterless_fig_candidates[:20]:
    print(f'  {p}')
## END for p in clusterless_fig_candidates[:20]...

if len(clusterless_csv_candidates) == 0:
    print('No clusterless train/test CSVs found for side-by-side comparison yet.')

## Export concatenated across-session CSVs

In [ ]:
export_map = {
    'bapun_train_test_decoder_error_df': bapun_train_test_decoder_error_df,
    'bapun_train_test_decoder_error_agg_df': bapun_train_test_decoder_error_agg_df,
    'bapun_context_decoder_laps_df': bapun_context_decoder_laps_df,
    'bapun_context_decoder_summary_agg_df': bapun_context_decoder_summary_agg_df,
    'most_recent_parsed_files_df': most_recent_parsed_files_df,
}

exported_paths = {}
for name, a_df in export_map.items():
    out_path = across_sessions_output_folder.joinpath(f'{BATCH_DATE_TO_USE}_{name}.csv').resolve()
    a_df.to_csv(out_path, index=False)
    exported_paths[name] = out_path
    print(f'wrote {name}: {out_path} ({len(a_df)} rows)')
## END for name, a_df in export_map.items()...

exported_paths